In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
rl = 50000
model = PPO.load(f"RLmodels/ppo_2566_{rl}_steps.zip", device="cpu")
model

In [3]:
fragment_pool = pd.read_csv("source/SMILES_list.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=2,
                          max_k=5,
                          max_steps=6,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [4]:
df = generate_SMILES(model, eval_env, n_episodes=100)
df.to_csv(f"generated_OPSs/rl_candidates_{rl}.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,Nc1cc(-c2c(Cl)noc2-c2[nH]ncc2Cl)c(-c2cc(-c3cnc...,"(Nc1cccnc1Cl, Cc1ccon1, Clc1cn[nH]c1, Clc1nccc...",2.497304
1,1,Cn1nc(-c2ccc3c(c2)N(C(C(=O)N2c4ccccc4C=Cc4cccc...,"(Nc1ccc([N+](=O)[O-])cn1, O=CCN1C(=O)COc2ccccc...",24.948691
2,2,Cn1nc(-c2c(F)cccc2Cl)c(-c2nc(C#N)c(-c3ccc4c(c3...,"(N#Cc1cocn1, Cn1nccc(N)c1=O, Fc1cccc(Cl)c1, O=...",6.286147
3,3,N#Cc1cc(-c2c(O)c(N)cc3c(-c4cc(Cl)c(Cl)cc4S)c(C...,"(Sc1ccc(Cl)c(Cl)c1, N#Cc1cnc2cc(O)c(N)cc2c1, N...",5.490650
4,4,Cc1cc(-c2nc(-c3cc(C#N)ccc3C)nc(-c3c([N+](=O)[O...,"(Cc1ccc(F)cc1Cl, Cc1cncnc1, Cc1ccc(C#N)cc1, Nc...",2.824303
...,...,...,...,...
95,95,Cn1c(-c2cc3n(n2)CCNC3=O)nc(-c2cc([N+](=O)[O-])...,"(Cn1ccnc1S, O=CN/N=C\c1ccccc1, O=[N+]([O-])c1c...",14.803144
96,96,C[S+]([O-])c1ccc(-c2ccc(N)cc2S(=O)(=O)c2cn3c([...,"(C[S+]([O-])c1ccccc1, Nc1cccc([SH](=O)=O)c1, O...",5.765920
97,97,Nc1ccc2c(c1)NCC(=Cc1nnc(Sc3cc(-c4cnc(S(N)(=O)=...,"(Nc1ccc2c(c1)NCCC2, Cc1nnc(S)s1, Nc1cccnc1Cl, ...",3.374069
98,98,N#Cc1nc(-c2cc3ccccc3oc2=N)oc1-c1c([N+](=O)[O-]...,"(N=c1ccc2ccccc2o1, N#Cc1cocn1, Nc1ccc([N+](=O)...",18.229593
